# Lantern LAGN Second Pass Classifier

*Notebook written by Natascha Barac*

*Based on code by Zach Gillis*

*Date last updated: 28 May 2026*

This notebook prepares our *Second Pass* classifier, which takes *First Cut* targets and filters them further down to LAGN candidates. This includes engineering a set of target features in order to distinguish between false positive targets, and true LAGN candidates that we want to focus on for further follow up. 

This notebook is meant to be run on the cloud-based Rubin Science Platform JupyterLab server. (This was last tested on RSP release 29.2.0). Much of the code for training and plotting the XGBoost classifier is taken from the `lantern_first_cut_filter.ipynb` notebook.

## Overview

After the *First Cut*, our goal is to further filter our sample of targets down to candidates. This *Second Pass* will feature a new set of features that were not used in the *First Cut*. In particular, we can begin to think about clustering in time and space in order to use features like spatial multiplicity, time variability, and persistence to make a higher purity sample of lensed AGN candidates.

Lensed AGN are predicted to appear and be efficiently detected as spatially extended sources in difference images from optical imaging surveys (Kochanek et al. 2006). This motivates our use of extendedness metrics as the primary discriminating features. This investigation is still in its preliminary stages. Notes about potential further explorations are included below. 

## Dataset

This *Second Pass* training can use the same training set detailed in the `lantern_first_cut_filter` training notebook. We then take that dataset, pass it through the *First Cut* filter, and use the output targets (labelled `True` LAGN by the filter) as the training set for our *Second Pass*. Ideally, we would use a completely separate dataset, but as we want to maximise the number of targets used in *Second Pass* training, we have decided to reuse the *First Pass* dataset. We note that the training and test sets were held separate in characterising the *First Cut*. 

In addition, because the *First Cut* is effective at cutting out the DP1 DIASources in our training set, using only this original `combined_training_data_v2.0.7.csv` dataset gave us only a very small number of non-LAGN in our *Second Pass* training. For this reason, we have also included more survey fields in our *Second Pass* training set. This can be adjusted in the `lantern_training_data_processing.ipynb` notebook. 

| Field | RA | Dec | Notes |
|---|---|---|---|
| ECDFS | 53.16° | −28.10° | Extended Chandra Deep Field South |
| Galactic | 95.0° | −25.0° | Low galactic latitude field |
| Ecliptic | 37.98° | 7.015° | Low ecliptic latitude field |

Note also that these datasets have already been clustered into sky positions of 3arcsec when the training set is created. This step will have to be done independently when working with real alerts. 

## Model

This notebook trains an XGBoost binary classifier to distinguish LAGN DIASources from non-LAGN DIASources, evaluates it on a held-out test set, and prepares a completeness/purity curve using estimated LSST survey statistics.

## Summary

1. **Load data** — read in combined_training_data_v{version}.csv and filter using the *First Cut* filter to build the final dataset
2. **Characterise** — visualise and characterise key properties of *Second Pass* training data, and investigate simple engineered features
3. **Train & Evaluate *Second Pass*** — train model and rescale to full survey size

---

## 1. Imports & Data Loading

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml
import corner

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

import xgboost

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)

import data_processing as dp

from sklearn.calibration import calibration_curve

import importlib
import data_processing as dp
importlib.reload(dp)
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

### Load Data & Model

In [ ]:
training_data = 'combined_training_data_more_DP1_all_cols.csv'

FEATURES = [
    # Other
    'band',
    # 'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    # 'trailFlux',
    # 'trailLength',
    
    # Dipole
    # 'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    # 'dipoleFluxDiffErr',
    # 'dipoleMeanFlux',
    # 'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

combined_training = pd.read_csv(training_data)

combined_training['isDipole'] = combined_training['isDipole'] == 'True'
combined_training['band'] = pd.Categorical(combined_training['band'])
band_categories = combined_training['band'].cat.categories

In [ ]:
import pickle

model_file = 'lantern_xgboost_t2.0.7.pkl'

with open(model_file, 'rb') as file:
    lantern_model = pickle.load(file)

### Build Filtered *Second Pass* Training Data

We want to test the filter on our full training set, to create another training set for our *Second Pass*. Note that, as mentioned previously, the `Lantern` filter has already seen this data, so there is some data leakage. A final version of this training would use data that has not been seen before. However, we want to maximise the data that we can use to train the *Second Pass*. 

In addition, the most important part of preventing data leakage is in the training of the model itself.

Here, we use ALL the alerts associated with a target that has at least one tag. This allows us to consider colour and other variable features (e.g. persistence) more extensively to distinguish between candidates and non-candidates. Moreover, it is more aligned with what we will get out of the tagged ANTARES Loci: we do not know which alerts individually passed the filter.

In [ ]:
X = combined_training.drop(columns=['label']) #[FEATURES]
y = combined_training['label']
groups = combined_training['lens_id']

print(f"  Full dataset: {len(X):,} DIAsources, {groups[y==1].nunique():,} lenses, {groups[y==0].nunique():,} non-lenses")

In [ ]:
threshold = 0.9691  #0.95 completeness
y_prob = lantern_model.predict_proba(X[FEATURES])[:,1]
y_pred = [1 if prob >= threshold else 0 for prob in y_prob]

In [ ]:
X['y_true'] = y
X['y_pred'] = y_pred
X['lens_id'] = groups

categories = []

#very slow, certainly a better way to do this—-just a stopgap for now!
for key, row in X.iterrows():
    if row['y_true'] == 0:
        if row['y_pred'] == 0:
            label = 'tn'
        elif row['y_pred'] == 1:
            label = 'fp'
    elif row['y_true'] == 1:
        if row['y_pred'] == 0:
            label = 'fn'
        elif row['y_pred'] == 1:
            label = 'tp'
    categories.append(label)

X['category'] = categories

In [ ]:
# tagged_alerts = all alerts tagged by First Cut filter
# targets = all alerts associated with a target that has at least one tagged alert

tagged_alerts = X[X.category.isin(['tp', 'fp'])]
targets = X[X.lens_id.isin(np.unique(tagged_alerts.lens_id))]

In [ ]:
print('TARGET INFO (alerts which pass First Cut)')
print('---------------------------------------------')

print(f'Total DIASources: {len(tagged_alerts)}')

print(f'\nNum. Unique Lenses: {tagged_alerts[tagged_alerts.lens_id > 0].lens_id.nunique()}')
print(f'Num. Lens DIASources (Tagged): {len(tagged_alerts[tagged_alerts.lens_id > 0])}')
print(f'Num. Lens DIASources (Total): {len(X[X.lens_id.isin(np.unique(tagged_alerts[tagged_alerts.lens_id > 0].lens_id))])}')

print(f'\nNum. Unique Non-Lenses: {tagged_alerts[tagged_alerts.lens_id < 0].lens_id.nunique()}')
print(f'Num. Non-Lens DIASources (Tagged): {len(tagged_alerts[tagged_alerts.lens_id < 0])}')
print(f'Num. Non-Lens DIASources (Total): {len(X[X.lens_id.isin(np.unique(tagged_alerts[tagged_alerts.lens_id < 0].lens_id))])}')

## 2. Characterising Targets for *Second Pass*
Quick look at general properties, in addition to first investigations for engineered features!

In [ ]:
tp_lens_ids = np.unique(X[X.category=='tp'].lens_id)
fn_lens_ids = np.unique(X[X.category=='fn'].lens_id)

completely_missed = [lens for lens in fn_lens_ids if lens not in tp_lens_ids]

print(f'Completely missed by filter: {len(completely_missed)}')
print(f'Total unique lenses (in full dataset): {len(np.unique(X[X.y_true==1].lens_id))}')

print(f'Fraction missed: {len(completely_missed)/len(np.unique(X[X.y_true==1].lens_id))}')

### Looking at tagged alerts

Looking at the AB magnitude of the tagged alerts, we see that the lenses which are completely missed by the *First Cut* filter are at the less-bright end of the spectrum. Notably, the false positive non-LAGN from DP1 are also on the less-bright end, however, they fall within the peak of the true LAGN distribution. Notably, we see good distinction between the true LAGN and the imposter non-LAGN in terms of the number of tagged alerts. 

In [ ]:
def flux_to_mag(flux):
    return -2.5 * np.log10(np.clip(flux, 1e-12, None)) + 31.4

bins = np.linspace(14, 30, 60)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6, 4), dpi=300, sharex=True)

true_pos = X[X['category'] == 'tp']  #tagged TP alerts
completely_missed_fn = X[(X['category'] == 'fn') & (X['lens_id'].isin(completely_missed))]

for ax, subset, title in [(ax0, true_pos, f'True Positive, all tagged DIASources (nLenses={len(np.unique(true_pos.lens_id))})'), (ax1, completely_missed_fn, f'Completely Missed Lenses, all DIASources (nLenses={len(np.unique(completely_missed_fn.lens_id))})')]:
    # subset = X[X['category'] == label]
    data_by_band = [
        flux_to_mag(subset.loc[subset['band'] == b, 'template_flux'].dropna().values)
        for b in dp.BAND_ORDER
    ]
    ax.hist(data_by_band, bins=bins, stacked=True,
            color=[dp.BAND_COLORS[b] for b in dp.BAND_ORDER], label=dp.BAND_ORDER, alpha=0.85, edgecolor='none')
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(title='band', fontsize=9, loc='center right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_xaxis()

ax1.set_xlabel('AB Magnitude of Template', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
bins = np.linspace(14, 30, 60)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6, 4), dpi=300, sharex=True)

true_pos = X[X['category'] == 'tp']
false_pos = X[X['category'] == 'fp']

for ax, subset, title in [(ax0, true_pos, f'True Positive, all tagged DIASources (nLenses={len(np.unique(true_pos.lens_id))})'), (ax1, false_pos, f'False Positive, all tagged DIASources (nLenses={len(np.unique(false_pos.lens_id))})')]:
    # subset = X[X['category'] == label]
    data_by_band = [
        flux_to_mag(subset.loc[subset['band'] == b, 'template_flux'].dropna().values)
        for b in dp.BAND_ORDER
    ]
    ax.hist(data_by_band, bins=bins, stacked=True,
            color=[dp.BAND_COLORS[b] for b in dp.BAND_ORDER], label=dp.BAND_ORDER, alpha=0.85, edgecolor='none')
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(title='band', fontsize=9, loc='center right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_xaxis()

ax1.set_xlabel('AB Magnitude of Template', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
diasources_per_id = tagged_alerts.groupby('lens_id').size()

bins = np.arange(diasources_per_id.min(), diasources_per_id.max() + 2) - 0.5

tps = tagged_alerts[tagged_alerts.category == 'tp'].groupby('lens_id').size()
fps = tagged_alerts[tagged_alerts.category == 'fp'].groupby('lens_id').size()

plt.rcParams['hatch.linewidth'] = 0.5

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(tps, bins=bins, color='forestgreen', edgecolor='white', linewidth=0.5, label='TP (injected LAGN)', alpha=0.5)
ax.hist(fps, bins=bins, color='slategray', edgecolor='white', linewidth=0.5, hatch="xxxx", label='FP (DP1 non-lenses)', alpha=0.5)

ax.set(xlabel='Alerts (DIASources) per Object (pass filter)', ylabel='Count')
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.legend()

plt.xlim(0,60)
plt.show()

### Looking at ALL alerts for Targets, and exploring engineered features

Because most of the non-LAGN have so few tagged alerts, it is difficult to identify any key properties of variability that distinguish between LAGN and non-LAGN. That is to say: any property that identifies variability will necessarily be zero for a target that has only one tagged alert, if we look only at that tagged alert. 

Simply making a separation based on the number of tagged alerts may be a good way of getting a high-purity sample of lenses from the Targets identified by the *First Cut*. However, note that we have not yet explicitly included imposters in our training data, which may not correspond to this simple cut. We therefore explore options looking at ALL alerts for targets, to examine how features that track variability between epochs perform. 

In [ ]:
bins = np.linspace(14, 30, 60)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6, 4), dpi=300, sharex=True)

lagn = targets[(targets['category']=='tp') | (targets['category']=='fn')]
non_lagn = targets[(targets['category']=='tn') | (targets['category']=='fp')]

for ax, subset, title in [(ax0, lagn, f'True LAGN Targets (nLenses={len(np.unique(lagn.lens_id))})'), (ax1, non_lagn, f'non-LAGN Targets (n_nonLenses={len(np.unique(non_lagn.lens_id))})')]:
    # subset = X[X['category'] == label]
    data_by_band = [
        flux_to_mag(subset.loc[subset['band'] == b, 'template_flux'].dropna().values)
        for b in dp.BAND_ORDER
    ]
    ax.hist(data_by_band, bins=bins, stacked=True,
            color=[dp.BAND_COLORS[b] for b in dp.BAND_ORDER], label=dp.BAND_ORDER, alpha=0.85, edgecolor='none')
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(title='band', fontsize=9, loc='center right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_xaxis()

ax1.set_xlabel('AB Magnitude of Template', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
tp_ids = np.unique(lagn.lens_id)
fp_ids = np.unique(non_lagn.lens_id)

diasources_per_id = X.groupby('lens_id').size()

bins = np.arange(diasources_per_id.min(), diasources_per_id.max() + 2) - 0.5

tps = lagn.groupby('lens_id').size()
fps = non_lagn.groupby('lens_id').size()

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(tps, bins=bins, color='forestgreen', edgecolor='white', linewidth=0.5, label='TP (injected LAGN)', alpha=0.5)
ax.hist(fps, bins=bins, color='slategray', edgecolor='white', linewidth=0.5, hatch='xxxx', label='FP (DP1 non-lenses)', alpha=0.5)

ax.set(xlabel='Alerts (DIASources) per Object (ALL ALERTS)', ylabel='Count')
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.legend()

plt.xlim(0,60)
plt.show()

It is notable that we include all alerts for our targets, there is no longer a clear separation between the LAGN and non-LAGN classes. However, engineering a "fraction tagged" feature allows us to see that true LAGN generally have a significantly larger fraction of alerts tagged than non-LAGN.

In [ ]:
grouped = targets[['templateFlux', 'lens_id']].groupby('lens_id').mean()

n_tagged = []
frac_tagged = []

for key, row in grouped.iterrows():
    lid = key
    n_tags = len(tagged_alerts[tagged_alerts.lens_id == lid])
    n_tagged.append(n_tags)
    
    n_alerts = len(targets[targets.lens_id == lid])
    ratio = n_tags / n_alerts * 100
    frac_tagged.append(np.round(ratio, decimals=5))

grouped['n_tagged'] = n_tagged
grouped['percent_tagged'] = frac_tagged

In [ ]:
# Set up bins based on the data range
bins = np.linspace(0, grouped['percent_tagged'].quantile(0.99), 40)
# bins = np.linspace(0, 30, 100)

plt.rcParams['hatch.linewidth'] = 0.5

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(grouped[grouped.index > 0]['percent_tagged'], 
        bins=bins, color='forestgreen', edgecolor='white', linewidth=0.5, label='Injected LAGN', alpha=0.6)
ax.hist(grouped[grouped.index < 0]['percent_tagged'], 
        bins=bins, color='slategrey', edgecolor='white', linewidth=0.5, hatch="xxxx", label='DP1 Non-Lenses', alpha=0.6)

ax.set(xlabel='Percent of Alerts Tagged', ylabel='Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.legend()
# ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Adjust x-limit as needed based on your data
# plt.xlim(0, 5)  # uncomment and adjust if needed

plt.title('Tagged alert persistence for First Cut Targets \n')
plt.show()

Centroid properties are another feature where we expect there to be significant difference between LAGN and non-LAGN. Here we look at centroid standard deviation, and centroid "instability" as measures of how much the centroid moves around between epochs.

We also calculate means and standard deviations for flux magnitudes, in order to examine how variability can be used to separate LAGN candidates from non-LAGN. This requires more investigation. We also expect colour to be a significant feature in separating between these two classes. 

In [ ]:
def calculate_stats(df):
    """
    Calculate centroid statistics and band-specific flux statistics for each unique object.
    
    Returns DataFrame with one row per unique object with centroid statistics and per-band flux measurements
    """
    
    # Group by object identifier
    grouped = df.groupby('lens_id')
    
    summary = grouped.agg(
        n_detections=('x', 'count'),
        x_std=('x', 'std'),
        y_std=('y', 'std'),
        mean_x_y_err=('x_y_err', 'mean'),
        median_x_y_err=('x_y_err', 'median'),

        # Global (not band-specific) statistics
        moment_ext_mean=('moment_ext', 'mean'),
        moment_ext_std=('moment_ext', 'std'),
        extendedness_mean=('extendedness', 'mean'),
        extendedness_std=('extendedness', 'std'),
        dipoleLength_mean = ('dipoleLength', 'mean'),
        dipoleLength_std = ('dipoleLength', 'std'),

        #Constant characteristics
        # category=('category', 'first'),
        y_true=('y_true', 'first'),
        ra=('ra', 'first'),
        dec=('dec', 'first'),

    ).reset_index()
    
    # Calculate combined centroid standard deviation
    summary['centroid_std'] = np.sqrt(summary['x_std']**2 + summary['y_std']**2)
    
    # Normalize by measurement error (instability metric)
    summary['centroid_instability'] = np.where(
        summary['median_x_y_err'] > 0,
        summary['centroid_std'] / summary['median_x_y_err'],
        0.0
    )
    
    # Handle single-detection objects (can't calculate std)
    summary['centroid_std'] = summary['centroid_std'].fillna(0.0)
    summary['centroid_instability'] = summary['centroid_instability'].fillna(0.0)
    
    # Calculate per-band statistics
    bands = ['u', 'g', 'r', 'i', 'z', 'y']
    
    for band in bands:
        # Filter to band
        band_data = df[df['band'] == band].groupby('lens_id').agg(
            **{
                f'apFlux_mean_{band}': ('apFlux', 'mean'),
                f'apFlux_std_{band}': ('apFlux', 'std'),
                f'template_flux_mean_{band}': ('template_flux', 'mean'),
                f'flux_ext_mean_{band}': ('flux_ext', 'mean'),
                f'flux_ext_std_{band}': ('flux_ext', 'std'),
            }
        )
        
        summary = summary.merge(band_data, left_on='lens_id', right_index=True, how='left')
    
    # Fill NaN for bands with no detections
    for band in bands:
        summary[f'apFlux_mean_{band}'] = summary[f'apFlux_mean_{band}'].fillna(np.nan)
        summary[f'apFlux_std_{band}'] = summary[f'apFlux_std_{band}'].fillna(np.nan)
        summary[f'template_flux_mean_{band}'] = summary[f'template_flux_mean_{band}'].fillna(np.nan)
        summary[f'flux_ext_mean_{band}'] = summary[f'flux_ext_mean_{band}'].fillna(np.nan)
        summary[f'flux_ext_std_{band}'] = summary[f'flux_ext_std_{band}'].fillna(np.nan)
    
    return summary

summary_df = calculate_stats(targets)

# columns :
# apFlux_mean_u, apFlux_std_u, template_flux_mean_u, flux_ext_mean_u,
# apFlux_mean_g, apFlux_std_g, template_flux_mean_g, flux_ext_mean_g,
# ... (for all bands)

In [ ]:
# Set up bins based on the data range
bins = np.linspace(0, summary_df['centroid_std'].quantile(0.99), 40)
# bins = np.linspace(0, 30, 100)

plt.rcParams['hatch.linewidth'] = 0.5

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(summary_df[summary_df['lens_id'] > 0]['centroid_std'], 
        bins=bins, color='forestgreen', edgecolor='white', linewidth=0.5, label='Injected LAGN', alpha=0.7)
ax.hist(summary_df[summary_df['lens_id'] < 0]['centroid_std'], 
        bins=bins, color='slategrey', edgecolor='white', linewidth=0.5, hatch="xxxx", label='DP1 Non-Lenses', alpha=0.7)

ax.set(xlabel='Centroid St. Dev.', ylabel='Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.legend()
# ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Adjust x-limit as needed based on your data
# plt.xlim(0, 5)  # uncomment and adjust if needed

plt.title('Centroid St.Dev for Targets\n')
plt.show()

In [ ]:
# Set up bins based on the data range
bins = np.linspace(0, summary_df['centroid_instability'].quantile(0.99), 40)
# bins = np.linspace(0, 30, 100)

plt.rcParams['hatch.linewidth'] = 0.5

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(summary_df[summary_df['lens_id'] > 0]['centroid_instability'], 
        bins=bins, color='forestgreen', edgecolor='white', linewidth=0.5, label='Injected LAGN', alpha=0.7)
ax.hist(summary_df[summary_df['lens_id'] < 0]['centroid_instability'], 
        bins=bins, color='slategrey', edgecolor='white', linewidth=0.5, hatch="xxxx", label='DP1 Non-Lenses', alpha=0.7)

ax.set(xlabel='Centroid Instability', ylabel='Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.legend()
# ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Adjust x-limit as needed based on your data
# plt.xlim(0, 5)  # uncomment and adjust if needed

plt.title('Centroid Instability per Target\n')
plt.show()

## 3. Model Training

Finally, we train a `Second Pass` model using these engineered features to see how well it performs on a first attempt. This also allows us to see which features become most important for the model. We remind the reader that this is still an initial pass, and further work is needed to refine these features, especially as we are now receiving real tagged alerts from ANTARES and can play around with the engineered features. 

What follows is a training process and evaluation which is very similar to that from the `lantern_first_cut_filter.ipynb` notebook.

In [ ]:
n_tagged = []
ratio_tagged = []

for key, row in summary_df.iterrows():
    lid = row['lens_id']
    n_tags = len(tagged_alerts[tagged_alerts.lens_id == lid])
    n_tagged.append(n_tags)
    
    n_alerts = len(X[X.lens_id == lid])
    ratio = n_tags / n_alerts * 100
    ratio_tagged.append(np.round(ratio, decimals=5))

summary_df['n_tagged'] = n_tagged
summary_df['percent_tagged'] = ratio_tagged

summary_df = summary_df.drop(columns=['n_detections'])

In [ ]:
X_model = summary_df.drop(columns=['lens_id', 'y_true', 'ra', 'dec']) #'category',
y_model = summary_df.y_true

# X_model = summary_df_no_band.drop(columns=['lens_id', 'category', 'y_true', 'ra', 'dec'])
# y_model = summary_df_no_band.y_true

from sklearn.model_selection import train_test_split

# Step 1: Split into 80% temp (for train/val) and 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)

# Step 2: Split temp into 75% train and 25% validation (resulting in 60% train, 20% val, 20% test)
X_train, X_eval, y_train, y_eval = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

### Hyperparameter Search

In [ ]:
param_grid = {
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.01, 0.02, 0.03],
    'subsample':        [0.6, 0.8],
    'colsample_bytree': [0.7, 0.9],
}

search = GridSearchCV(
    XGBClassifier(
        n_estimators=500,
        # scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    refit=False,
    verbose=1,
)
search.fit(X_train, y_train, sample_weight=sample_weight_train)

best_params = search.best_params_
print(f"\nBest params: {best_params}")

### Train & Evaluate Model

In [ ]:
X_train.columns

In [ ]:
MANUAL_PARAMS = {
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.9,
}

# params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS
params = MANUAL_PARAMS

model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, #sample_weight=sample_weight_train,
          eval_set=[(X_eval, y_eval)], verbose=False)
print(f"Training complete. Best n_estimators: {model.best_iteration}")

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-LAGN (0)', 'LAGN (1)'], digits=4))

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# Feature Importance
ax1 = fig.add_subplot(gs[0, 0])
importance = model.feature_importances_
feat_names = np.array(X_model.columns.tolist())
top_idx = np.argsort(importance)[-20:]
ax1.barh(feat_names[top_idx], importance[top_idx], color='steelblue', edgecolor='white')
ax1.set_xlabel('Gain', fontsize=11)
ax1.set_title('Feature Importance', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelsize=8)
ax1.spines[['top', 'right']].set_visible(False)

# ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost\n(AUC = {roc_auc_val:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.02])
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)

# Precision-Recall Curve
ax3 = fig.add_subplot(gs[0, 2])
pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
baseline = y_test.mean()
ax3.plot(pr_recall, pr_precision, color='darkorange', lw=2, label=f'XGBoost\n(AP = {ap:.4f})')
ax3.axhline(baseline, color='navy', lw=1.5, linestyle='--',
            label=f'Random\n(AP = {baseline:.4f})')
ax3.fill_between(pr_recall, pr_precision, alpha=0.08, color='darkorange')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.02])
ax3.set_xlabel('Recall', fontsize=11)
ax3.set_ylabel('Precision', fontsize=11)
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.spines[['top', 'right']].set_visible(False)

# Predicted Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
bins = np.linspace(0, 1, 60)
ax4.hist(y_prob[y_test == 0], bins=bins, alpha=0.85, color='slategray',
         label='Non-LAGN (0)', density=True)
ax4.hist(y_prob[y_test == 1], bins=bins, alpha=0.85, color='forestgreen',
         label='LAGN (1)', density=True)
ax4.axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax4.set_xlabel('P(LAGN)', fontsize=11)
ax4.set_ylabel('Density', fontsize=11)
ax4.set_title('P(LAGN)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10, loc='upper center')
ax4.spines[['top', 'right']].set_visible(False)

# Confusion Matrix Heatmap
ax5 = fig.add_subplot(gs[1, 1])
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels_raw = [f'{v:,}' for v in cm.ravel()]
labels_pct = [f'{v:.1%}' for v in cm_norm.ravel()]
annot = np.array([f'{r}\n({p})' for r, p in zip(labels_raw, labels_pct)]).reshape(2, 2)

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax5,
            xticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            yticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Row-normalised rate'})
ax5.set_xlabel('Predicted Label', fontsize=11)
ax5.set_ylabel('True Label', fontsize=11)
ax5.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.show()

In [ ]:
# Print feature importance
sorted_idx = np.argsort(importance)[::-1]
print('Full feature importance:')
for rank, i in enumerate(sorted_idx, 1):
    print(f"{rank:>3}. {feat_names[i]:<35} {importance[i]:.6f}")                                      

In [ ]:
CLUSTER_FEATURES = ['x_std', 'y_std', 'mean_x_y_err', 'median_x_y_err',
       'moment_ext_mean', 'moment_ext_std', 'extendedness_mean',
       'extendedness_std', 'dipoleLength_mean', 'dipoleLength_std', 'y_true',
       'ra', 'dec', 'centroid_std', 'centroid_instability', 'apFlux_mean_u',
       'apFlux_std_u', 'template_flux_mean_u', 'flux_ext_mean_u',
       'flux_ext_std_u', 'apFlux_mean_g', 'apFlux_std_g',
       'template_flux_mean_g', 'flux_ext_mean_g', 'flux_ext_std_g',
       'apFlux_mean_r', 'apFlux_std_r', 'template_flux_mean_r',
       'flux_ext_mean_r', 'flux_ext_std_r', 'apFlux_mean_i', 'apFlux_std_i',
       'template_flux_mean_i', 'flux_ext_mean_i', 'flux_ext_std_i',
       'apFlux_mean_z', 'apFlux_std_z', 'template_flux_mean_z',
       'flux_ext_mean_z', 'flux_ext_std_z', 'apFlux_mean_y', 'apFlux_std_y',
       'template_flux_mean_y', 'flux_ext_mean_y', 'flux_ext_std_y', 'n_tagged',
       'percent_tagged']

In [ ]:
# SELECT FEATURES TO PLOT
SELECTED_FEATURES = [
    'x_std', 
    'y_std', 
    # 'mean_x_y_err', 
    # 'median_x_y_err',
    # 'moment_ext_mean', 
    # 'moment_ext_std', 
    # 'extendedness_mean',
    # 'extendedness_std', 
    # 'dipoleLength_mean', 
    # 'dipoleLength_std',
    'centroid_std', 
    # 'centroid_instability', 
    # 'apFlux_mean_u',
    # 'apFlux_std_u', 
    # 'template_flux_mean_u', 
    # 'flux_ext_mean_u',
    'flux_ext_std_u', 
    # 'apFlux_mean_g', 
    # 'apFlux_std_g',
    # 'template_flux_mean_g', 
    # 'flux_ext_mean_g', 
    # 'flux_ext_std_g',
    # 'apFlux_mean_r', 
    # 'apFlux_std_r', 
    # 'template_flux_mean_r',
    # 'flux_ext_mean_r', 
    # 'flux_ext_std_r', 
    # 'apFlux_mean_i', 
    # 'apFlux_std_i',
    # 'template_flux_mean_i', 
    # 'flux_ext_mean_i', 
    # 'flux_ext_std_i',
    # 'apFlux_mean_z', 
    # 'apFlux_std_z', 
    # 'template_flux_mean_z',
    # 'flux_ext_mean_z', 
    # 'flux_ext_std_z', 
    # 'apFlux_mean_y', 
    # 'apFlux_std_y',
    # 'template_flux_mean_y', 
    # 'flux_ext_mean_y', 
    # 'flux_ext_std_y', 
    'n_tagged',
    'percent_tagged'
    # Add or remove features as desired
]

FEATURE_PROPS = {
    # Position and centroid features
    'x_std':                  ('linear', (0, 3000),        'X Std'),
    'y_std':                  ('linear', (0, 3000),        'Y Std'),
    'mean_x_y_err':           ('linear', (0, 4),        'Mean Cent. Err'),
    'median_x_y_err':         ('linear', (0, 4),        'Median Cent. Err'),
    'centroid_std':           ('linear', (0, 3000),        'Centroid Std'),
    'centroid_instability':   ('linear', (0, 10),       'Cent. Instability'),
    
    # Moment and extendedness features
    'moment_ext_mean':        ('linear', (0, 4),        'Moment Ext. Mean'),
    'moment_ext_std':         ('linear', (0, 2),        'Moment Ext. Std'),
    'extendedness_mean':      ('linear', (0, 1),        'Rubin Ext. Mean'),
    'extendedness_std':       ('linear', (0, 0.5),      'Rubin Ext. Std'),
    
    # Dipole features
    'dipoleLength_mean':      ('linear', (0, 0.15),     'Dip. Length Mean'),
    'dipoleLength_std':       ('linear', (0, 0.1),      'Dip. Length Std'),
    
    # Coordinates
    'ra':                     ('linear', (0, 360),      'RA (deg)'),
    'dec':                    ('linear', (-90, 90),     'Dec (deg)'),
    
    # u-band features
    'apFlux_mean_u':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (u)'),
    'apFlux_std_u':           ('log',    (1e1, 1e6),    'Ap. Flux Std (u)'),
    'template_flux_mean_u':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (u)'),
    'flux_ext_mean_u':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (u)'),
    'flux_ext_std_u':         ('linear', (0, 2),        'Flux Ext. Std (u)'),
    
    # g-band features
    'apFlux_mean_g':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (g)'),
    'apFlux_std_g':           ('log',    (1e1, 1e6),    'Ap. Flux Std (g)'),
    'template_flux_mean_g':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (g)'),
    'flux_ext_mean_g':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (g)'),
    'flux_ext_std_g':         ('linear', (0, 5),        'Flux Ext. Std (g)'),
    
    # r-band features
    'apFlux_mean_r':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (r)'),
    'apFlux_std_r':           ('log',    (1e1, 1e6),    'Ap. Flux Std (r)'),
    'template_flux_mean_r':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (r)'),
    'flux_ext_mean_r':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (r)'),
    'flux_ext_std_r':         ('linear', (0, 5),        'Flux Ext. Std (r)'),
    
    # i-band features
    'apFlux_mean_i':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (i)'),
    'apFlux_std_i':           ('log',    (1e1, 1e6),    'Ap. Flux Std (i)'),
    'template_flux_mean_i':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (i)'),
    'flux_ext_mean_i':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (i)'),
    'flux_ext_std_i':         ('linear', (0, 5),        'Flux Ext. Std (i)'),
    
    # z-band features
    'apFlux_mean_z':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (z)'),
    'apFlux_std_z':           ('log',    (1e1, 1e6),    'Ap. Flux Std (z)'),
    'template_flux_mean_z':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (z)'),
    'flux_ext_mean_z':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (z)'),
    'flux_ext_std_z':         ('linear', (0, 5),        'Flux Ext. Std (z)'),
    
    # y-band features
    'apFlux_mean_y':          ('log',    (1e2, 1e7),    'Ap. Flux Mean (y)'),
    'apFlux_std_y':           ('log',    (1e1, 1e6),    'Ap. Flux Std (y)'),
    'template_flux_mean_y':   ('log',    (1e2, 1e7),    'Temp. Flux Mean (y)'),
    'flux_ext_mean_y':        ('log',    (0.1, 10.0),   'Flux Ext. Mean (y)'),
    'flux_ext_std_y':         ('linear', (0, 5),        'Flux Ext. Std (y)'),
    
    # Tagging features
    'n_tagged':               ('linear', (0, 50),       'N Tagged'),
    'percent_tagged':         ('linear', (0, 100),      'Percent Tagged'),
}

def to_array(subset, cols):
    arr = np.empty((len(subset), len(cols)), dtype=float)
    for i, c in enumerate(cols):
        scale, (lo, hi) = FEATURE_PROPS[c][0], FEATURE_PROPS[c][1]
        s = pd.to_numeric(subset[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if scale == 'log':
            s = s.where(s > 0)      # mask non-positive: log(≤0) is undefined
        med = s.median()
        fill = med if np.isfinite(med) else lo   # fall back to range lower bound
        s = s.fillna(fill)
        arr[:, i] = s.values
        # Final safety: clamp any surviving bad values
        if scale == 'log':
            bad = ~np.isfinite(arr[:, i]) | (arr[:, i] <= 0)
            arr[bad, i] = lo
        else:
            arr[~np.isfinite(arr[:, i]), i] = 0.0
    return arr

df_false = summary_df[summary_df['y_true'] == 0]
df_true  = summary_df[summary_df['y_true'] == 1]

XGB_THRESHOLD = 0.9

all_probs = model.predict_proba(X_model)[:, 1]
df_xgb = summary_df[all_probs > XGB_THRESHOLD]
print(f"Sources passing XGBoost (threshold={XGB_THRESHOLD}): {len(df_xgb):,}")

# Filter candidate_cols to only include SELECTED_FEATURES
candidate_cols = [f for f in SELECTED_FEATURES if f in CLUSTER_FEATURES and f != 'band']
print(f"Using {len(candidate_cols)} selected features: {candidate_cols}")

# Drop columns where any population has fewer than 2 unique in-range values
def is_plottable(col, *dfs, min_samples=2):
    scale, (lo, hi) = FEATURE_PROPS[col][0], FEATURE_PROPS[col][1]
    for d in dfs:
        s = pd.to_numeric(d[col], errors='coerce')
        if scale == 'log':
            s = s.where(s > 0)
        vals = s.dropna()
        in_range = vals[(vals >= lo) & (vals <= hi)]
        if in_range.nunique() < min_samples:
            return False
    return True

columns_corner = [c for c in candidate_cols if is_plottable(c, df_false, df_true, df_xgb)]
dropped = set(candidate_cols) - set(columns_corner)
if dropped:
    print(f"Dropped unplottable columns: {dropped}")

print(f"Final plotting columns ({len(columns_corner)}): {columns_corner}")

axes_scale = [FEATURE_PROPS[f][0] for f in columns_corner]
ranges     = [FEATURE_PROPS[f][1] for f in columns_corner]
labels     = [FEATURE_PROPS[f][2] for f in columns_corner]

data_array_all  = to_array(df_false, columns_corner)
data_array_lagn = to_array(df_true,  columns_corner)
data_array_xgb  = to_array(df_xgb,   columns_corner)

# Adjust figure size based on number of features
n_features = len(columns_corner)
fig_size = max(8, min(15, n_features * 2.5))  # Scale between 8 and 15

fig = corner.corner(data_array_all,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=plt.figure(figsize=(fig_size, fig_size), dpi=300),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

# fig = corner.corner(data_array_lagn,
#                     labels=labels,
#                     axes_scale=axes_scale,
#                     range=ranges,
#                     fill_contours=True,
#                     smooth=0.7,
#                     show_titles=False,
#                     color='green',
#                     plot_datapoints=False,
#                     plot_contours=True,
#                     plot_density=True,
#                     bins=20,
#                     fig=fig,
#                     max_n_ticks=3,
#                     hist_kwargs=dict(density=True)
#                    )

fig = corner.corner(data_array_xgb,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='red',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True),
                    alpha=0.3
                   )

fig = corner.corner(data_array_lagn,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='green',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

for ax in fig.axes:
    ax.tick_params(labelsize=8, axis='both', which='major', pad=2)
    for label in ax.get_xticklabels():
        label.set_rotation(0)
    xlabels = [t for t in ax.xaxis.get_major_ticks() if t.label1.get_visible()]
    if xlabels:
        xlabels[-1].label1.set_visible(False)
    ylabels = [t for t in ax.yaxis.get_major_ticks() if t.label1.get_visible()]
    if ylabels:
        ylabels[0].label1.set_visible(False)

legend_elements = [
    Patch(facecolor='grey',  edgecolor='black',    label=f'DP1 DIASources Targets ({len(df_false):,})'),
    Patch(facecolor='green', edgecolor='darkgreen', label=f'Injected LAGN Targets ({len(df_true):,})'),
    Patch(facecolor='red',   edgecolor='darkred',   label=f'XGBoost p > {XGB_THRESHOLD} ({len(df_xgb):,})'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=13, framealpha=0.9)

plt.show()

### Train & Save Final Model

In [ ]:
# ── Train Final Model ──────────────────────────────────────────────────────────
USE_BEST_PARAMS = False

MANUAL_PARAMS = {
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.9,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

# Train with early stopping to find best n_estimators
print("Training with early stopping to find optimal n_estimators...")
model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, #sample_weight=sample_weight_train,
          eval_set=[(X_eval, y_eval)], verbose=False)
best_n_estimators = model.best_iteration
print(f"Best n_estimators: {best_n_estimators}")

# Retrain on train+eval+test with the optimal n_estimators (no early stopping)
print(f"\nRetraining final model on train+eval+test data with n_estimators={best_n_estimators}...")
X_train_final = pd.concat([X_train, X_eval, X_test])
y_train_final = pd.concat([y_train, y_eval, y_test])

final_model = XGBClassifier(
    n_estimators=best_n_estimators,  # Use the optimal value found
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    **params,
)

final_model.fit(X_train_final, y_train_final, verbose=False)
print("Final model training complete.")

In [ ]:
# # Save the final model
import pickle

training_version = '2.0.7'
model_filename = f'lantern_second_pass_xgboost_t{training_version}.pkl'

with open(model_filename, 'wb') as f:
    pickle.dump(final_model, f)

print(f"Model saved to {model_filename}")

### Survey Scale Analysis

In [ ]:
groups = summary_df.lens_id
groups_train = groups.loc[X_train.index]
groups_test = groups.loc[X_test.index]

In [ ]:
# these can be rescaled for the true yr1 data, which will cover the DP2 area
# we have been assuming a 1000 sq deg. area for DP2 (which we note will not be fully templated)

lagn_diasources     = 20_190 #number of LAGN DIASources coming out of FIRST CUT in Y2 (95% completeness threshold)
non_lagn_diasources = 5_380_529 #number of tagged non-LAGN DIASources in FIRST CUT targets in Y2 (95% completeness threshold)
N_lenses_yr1 = 2524 #number of unique LAGN in FIRST CUT targets in Y2 (95% c.)

# lagn_diasources     = 1124 #number of LAGN DIASources coming out of FIRST CUT in Y1 (95% completeness threshold)
# non_lagn_diasources = 297_844 #number of tagged non-LAGN DIASources in FIRST CUT targets in Y1 (95% completeness threshold)
# N_lenses_yr1 = 141 #number of unique LAGN in FIRST CUT targets in Y1 (95% c.)

N_nonlenses_full = tagged_alerts[tagged_alerts['y_true'] == 0]['lens_id'].nunique()
N_nonlagn_diasources_full = (tagged_alerts['y_true'] == 0).sum()
avg_diasources_per_nonlens = N_nonlagn_diasources_full / N_nonlenses_full
N_nonlenses_yr1 = int(non_lagn_diasources / avg_diasources_per_nonlens)

print(f"\nLAGN DIASources     (yr 1) : {lagn_diasources:,}")
print(f"Non-LAGN DIASources (yr 1) : {non_lagn_diasources:,}")
 
cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
comp_grid = np.linspace(0, 1, 100)
 
# Store metrics for both levels
fold_purity_sources = []
fold_purity_objects = []
fold_lens_frac = []
fold_nonlens_frac = []
 
for train_idx, val_idx in cv.split(X_train, y_train, groups_train):
    # fold_sw = compute_band_weights(y_stage2_train.iloc[train_idx], X_stage2_train.iloc[train_idx])
 
    fold_model = XGBClassifier(
        n_estimators=300,
        eval_metric='logloss', enable_categorical=True,
        random_state=42, n_jobs=-1, **params,
    )
    fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx],
                   verbose=False)
                   # sample_weight=fold_sw, verbose=False)
 
    # Get predicted probabilities and true labels for validation fold
    prob = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    labels = np.array(y_train.iloc[val_idx])
    val_lens_id = np.array(groups_train.iloc[val_idx])
 
    # Sort validation data by descending predicted probability
    order = np.argsort(prob)[::-1]
    labels_sorted = labels[order]
    lens_id_sorted = val_lens_id[order]
 
    # --------------DIASource-level metrics------------------------------
    N_pos_sources = (labels == 1).sum()
    N_neg_sources = (labels == 0).sum()
 
    TP_sources = np.cumsum(labels_sorted == 1)
    FP_sources = np.cumsum(labels_sorted == 0)
 
    tpr_sources = TP_sources / N_pos_sources
    fpr_sources = FP_sources / N_neg_sources
 
    # Scale to survey population
    TP_survey_sources = tpr_sources * lagn_diasources
    FP_survey_sources = fpr_sources * non_lagn_diasources
    purity_sources = TP_survey_sources / (TP_survey_sources + FP_survey_sources).clip(1e-12)
 
    fold_purity_sources.append(np.interp(comp_grid, tpr_sources, purity_sources))
 
    # --------------Unique object-level metrics------------------------------
    N_lenses_fold = len(np.unique(val_lens_id[labels == 1]))
    N_nonlenses_fold = len(np.unique(val_lens_id[labels == 0]))
 
    # Track unique objects recovered as we descend sorted list
    seen_lenses = set()
    seen_nonlenses = set()
    n_lenses_cumul = np.zeros(len(labels_sorted), dtype=int)
    n_nonlenses_cumul = np.zeros(len(labels_sorted), dtype=int)
 
    for j in range(len(labels_sorted)):
        lid = lens_id_sorted[j]
        if labels_sorted[j] == 1:
            seen_lenses.add(lid)
        else:
            seen_nonlenses.add(lid)
        n_lenses_cumul[j] = len(seen_lenses)
        n_nonlenses_cumul[j] = len(seen_nonlenses)
 
    tpr_objects = n_lenses_cumul / max(N_lenses_fold, 1)
    fpr_objects = n_nonlenses_cumul / max(N_nonlenses_fold, 1)
 
    # Scale to survey population
    TP_survey_objects = tpr_objects * N_lenses_yr1
    FP_survey_objects = fpr_objects * N_nonlenses_yr1
    purity_objects = TP_survey_objects / (TP_survey_objects + FP_survey_objects).clip(1e-12)
 
    fold_purity_objects.append(np.interp(comp_grid, tpr_objects, purity_objects))
 
    fold_lens_frac.append(tpr_objects[-1] if len(tpr_objects) > 0 else 0)
    fold_nonlens_frac.append(fpr_objects[-1] if len(fpr_objects) > 0 else 0)
 
# ----------------Aggregate results across folds------------------------------
mean_purity_sources = np.mean(fold_purity_sources, axis=0)
mean_purity_objects = np.mean(fold_purity_objects, axis=0)

In [ ]:
sample_size_yr1  = comp_grid * lagn_diasources
lenses_yr1       = mean_purity_objects * N_lenses_yr1

### Exploring Projections @ Candidate Level

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
ax.plot(comp_grid, mean_purity_objects, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity (Unique Objects, 10-fold CV)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Plot 2: Sample Composition vs. Purity @ Object Level
sample_size_yr1_objects = comp_grid * N_lenses_yr1
total_objects_yr1 = sample_size_yr1_objects / np.clip(mean_purity_objects, 1e-12, None)

fig, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
fig.subplots_adjust(right=0.75)
ax1.plot(mean_purity_objects, sample_size_yr1_objects, color='forestgreen', lw=2, label='Lenses from LAGN')
ax1.set_xscale('log')
ax1.set_xlim(1e-4, 1)
ax1.set_xlabel('Purity')
ax1.set_ylabel('Unique Lenses from LAGN (year 1)', color='forestgreen')
ax1.tick_params(axis='y', labelcolor='forestgreen')
ax1.spines[['top']].set_visible(False)
ax1.set_title('Sample Composition vs. Purity (year 1, unique objects)')
ax1.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)

ax2 = ax1.twinx()
ax2.spines['right'].set_position(('outward', 0))
ax2.plot(mean_purity_objects, total_objects_yr1, color='slategrey', lw=2, label='Total objects')
ax2.set_ylabel('Total unique objects in sample (year 1)', color='slategrey')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='slategrey')
ax2.spines[['top']].set_visible(False)

ax2.set_xlim(10e-3,1)

lines = [ax1.get_lines()[0], ax2.get_lines()[0]]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)
plt.show()